In [3]:
import cv2
import numpy as np
import requests
from PIL import Image
from io import BytesIO
import os
import threading
import time
import ipywidgets as widgets
from IPython.display import display, HTML

# Jupyter çıktı alanında slider'ı engelle
display(HTML("""
<style>
.output_scroll { height: auto !important; max-width: 100% !important; overflow: hidden !important; }
.widget-hbox { overflow: hidden !important; max-width: 100% !important; }
.widget-box { overflow: hidden !important; }
.container { width: 100% !important; }
.widget-image { border: 2px solid gray !important; }
</style>
"""))

# 📁 Dataset klasörü
dataset_folder = "dataset"
os.makedirs(dataset_folder, exist_ok=True)
image_files = [f for f in os.listdir(dataset_folder) if f.endswith((".png", ".jpg", ".jpeg"))]

DISPLAY_WIDTH, DISPLAY_HEIGHT = 450, 338  # Net görüntü, slider'sız

# 📷 Görüntü gösterim alanları
image_output_original = widgets.Image(
    layout={"width": f"{DISPLAY_WIDTH}px", "height": f"{DISPLAY_HEIGHT}px", "max_width": f"{DISPLAY_WIDTH}px", "object_fit": "contain", "border": "2px solid gray !important"}
)
image_output_processed = widgets.Image(
    layout={"width": f"{DISPLAY_WIDTH}px", "height": f"{DISPLAY_HEIGHT}px", "max_width": f"{DISPLAY_WIDTH}px", "object_fit": "contain", "border": "2px solid gray !important"}
)
image_container = widgets.HBox(
    [image_output_original, image_output_processed],
    layout={"overflow": "hidden", "max_width": f"{DISPLAY_WIDTH*2+4}px", "display": "flex", "flex_wrap": "nowrap", "gap": "4px"}
)

# 🎛️ Parametre sekmeleri
tab = widgets.Tab()

params_wb = widgets.VBox([
    widgets.IntSlider(value=50, min=0, max=255, description="Threshold:"),
    widgets.IntSlider(value=5000, min=1000, max=10000, description="Min Area:"),
    widgets.FloatSlider(value=2.0, min=0.5, max=5.0, description="CLAHE Clip:"),
    widgets.FloatSlider(value=0.1, min=0.01, max=0.8, description="Stripe Ratio:")
])

params_simple = widgets.VBox([
    widgets.IntSlider(value=50, min=0, max=255, description="Threshold:"),
    widgets.IntSlider(value=5000, min=1000, max=10000, description="Min Area:"),
    widgets.FloatSlider(value=2.0, min=0.5, max=5.0, description="CLAHE Clip:"),
    widgets.IntSlider(value=5, min=3, max=9, step=2, description="Kernel Size:")
])

params_batch = widgets.VBox([
    widgets.IntSlider(value=50, min=0, max=255, description="Threshold:"),
    widgets.FloatSlider(value=2.0, min=0.5, max=5.0, description="CLAHE Clip:"),
    widgets.IntSlider(value=5, min=3, max=9, step=2, description="Kernel Size:")
])

tab.children = [params_wb, params_simple, params_batch]
tab.set_title(0, "White-Black Stripe")
tab.set_title(1, "Simple Contour")
tab.set_title(2, "Batch Contour")

# 🔘 Resim seçimi
image_dropdown = widgets.Dropdown(
    options=image_files if image_files else ["Görüntü yok"],
    description="Resim:"
)
process_button = widgets.Button(description="Görüntü İşle", button_style='success')

# 🌍 API adresi
API_URL = "http://127.0.0.1:8000/api/capture-image/"

# 🔁 Preview kontrol
preview_stop = threading.Event()
preview_thread = None
last_frame = None  # Yakalanacak son kare

# 🖼️ Görüntü gösterme
def show_image(img_array, target="original"):
    try:
        print(f"show_image çağrıldı, hedef: {target}, frame tipi: {type(img_array)}, boyut: {img_array.shape}")
        img_rgb = cv2.cvtColor(img_array, cv2.COLOR_BGR2RGB)
        print(f"RGB dönüşüm sonrası: {img_rgb.shape}")
        img_resized = cv2.resize(img_rgb, (DISPLAY_WIDTH, DISPLAY_HEIGHT))
        pil_image = Image.fromarray(img_resized)
        buffered = BytesIO()
        pil_image.save(buffered, format="JPEG")
        img_data = buffered.getvalue()
        output_widget = image_output_original if target == "original" else image_output_processed
        output_widget.value = img_data
        print(f"Görüntü widget'a yazıldı, hedef: {target}, veri boyutu: {len(img_data)}")
    except Exception as e:
        print(f"⚠️ Görüntü gösterme hatası: {e}")

# 🚨 Hata mesajı gösterme
def show_error(message, target="original"):
    error_img = np.zeros((DISPLAY_HEIGHT, DISPLAY_WIDTH, 3), dtype=np.uint8)
    cv2.putText(error_img, message, (10, DISPLAY_HEIGHT//2), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    show_image(error_img, target=target)

# 🎞️ Webcam preview
def start_webcam_preview():
    global last_frame
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("⚠️ Webcam açılamadı. Cihaz kontrol ediliyor...")
        show_error("Webcam açılamadı.")
        return
    print("Webcam başarıyla açıldı.")
    last_update = time.time()
    while not preview_stop.is_set():
        ret, frame = cap.read()
        if ret:
            last_frame = frame.copy()
            fps = int(1 / (time.time() - last_update)) if time.time() - last_update > 0 else 0
            cv2.putText(frame, f"FPS: {fps}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
            show_image(frame, target="original")
            print(f"Webcam karesi alındı, boyut: {frame.shape}, zaman: {time.strftime('%H:%M:%S')}, FPS: {fps}")
            last_update = time.time()
        else:
            print("⚠️ Webcam karesi alınamadı.")
            show_error("Kare alınamadı.")
        time.sleep(0.033)  # ~30 FPS
    cap.release()
    print("Webcam akışı durduruldu.")
    image_output_original.value = b""

# 🌐 API preview
def start_api_preview():
    global last_frame
    API_URL = "http://127.0.0.1:8000/api/video-feed/"  # Doğru URL
    print("API akışı başlatılıyor...")
    last_update = time.time()
    while not preview_stop.is_set():
        try:
            response = requests.get(API_URL, timeout=20, stream=True)
            print(f"API yanıtı: {response.status_code}, zaman: {time.strftime('%H:%M:%S')}")
            if response.status_code == 200:
                boundary = response.headers.get('content-type').split('boundary=')[-1].encode()
                buffer = b""
                for chunk in response.iter_content(chunk_size=2048):  # Daha büyük chunk boyutu
                    if preview_stop.is_set():
                        break
                    buffer += chunk
                    while b'--' + boundary in buffer:
                        try:
                            frame_data, buffer = buffer.split(b'--' + boundary, 1)
                            if b'Content-Type: image/jpeg' in frame_data:
                                img_data = frame_data.split(b'\r\n\r\n', 1)[1].rsplit(b'\r\n', 1)[0]
                                img = Image.open(BytesIO(img_data))
                                frame = np.array(img)
                                if frame.shape[-1] == 4:  # RGBA to BGR
                                    frame = cv2.cvtColor(frame, cv2.COLOR_RGBA2BGR)
                                elif frame.shape[-1] == 3:  # RGB to BGR
                                    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
                                last_frame = frame.copy()
                                fps = int(1 / (time.time() - last_update)) if time.time() - last_update > 0 else 0
                                cv2.putText(frame, f"FPS: {fps}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
                                show_image(frame, target="original")
                                print(f"API karesi alındı, boyut: {frame.shape}, FPS: {fps}")
                                last_update = time.time()
                        except Exception as e:
                            print(f"⚠️ Kare işleme hatası: {e}")
                            continue
            else:
                print(f"❌ API yanıtı: {response.status_code}")
        except requests.exceptions.RequestException as e:
            print(f"⚠️ API hatası: {e}")
        time.sleep(0.1)  # ~10 FPS
    print("API akışı durduruldu.")
    image_output_original.value = b""

# 🌈 Ön işlem
def preprocess_image(img, clahe_clip):
    img_resized = cv2.resize(img, (960, 640))
    gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=(8, 8))
    gray_enhanced = clahe.apply(gray)
    return img_resized, gray_enhanced

# 🔍 Algoritmalar
def white_black_stripe_detection(img, gray, threshold, min_area, stripe_ratio):
    try:
        print("white_black_stripe_detection çalışıyor...")
        white_mask = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, -5)
        _, black_mask = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY_INV)
        black_stripes = cv2.bitwise_and(black_mask, white_mask)
        print(f"Kontur bulma öncesi, white_mask boyutu: {white_mask.shape}")
        contours, _ = cv2.findContours(white_mask, cv2.RETR_EXTERNAL, getattr(cv2, 'CHAIN_APROX_SIMPLE', 2))
        print(f"Kontur sayısı: {len(contours)}")
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if min_area < area < 30000:
                x, y, w, h = cv2.boundingRect(cnt)
                region = black_stripes[y:y+h, x:x+w]
                black_ratio = np.sum(region == 255) / (w * h)
                if 0.005 < black_ratio < stripe_ratio:
                    cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 4)
                    cv2.putText(img, "Robot", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        return img
    except Exception as e:
        print(f"⚠️ white_black_stripe_detection hatası: {e}")
        raise

def simple_contour_detection(img, gray, threshold, min_area, kernel_size):
    try:
        print("simple_contour_detection çalışıyor...")
        _, binary = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY_INV)
        kernel = np.ones((kernel_size, kernel_size), np.uint8)
        cleaned = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=2)
        print(f"Kontur bulma öncesi, cleaned boyutu: {cleaned.shape}")
        contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, getattr(cv2, 'CHAIN_APROX_SIMPLE', 2))
        print(f"Kontur sayısı: {len(contours)}")
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if min_area < area < 30000:
                x, y, w, h = cv2.boundingRect(cnt)
                cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 4)
                cv2.putText(img, "Robot", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                break
        return img
    except Exception as e:
        print(f"⚠️ simple_contour_detection hatası: {e}")
        raise

def batch_contour_detection(img, gray, threshold, kernel_size):
    try:
        print("batch_contour_detection çalışıyor...")
        _, binary = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_size, kernel_size))
        cleaned = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=2)
        cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel, iterations=2)
        print(f"Kontur bulma öncesi, cleaned boyutu: {cleaned.shape}")
        contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, getattr(cv2, 'CHAIN_APROX_SIMPLE', 2))
        print(f"Kontur sayısı: {len(contours)}")
        if contours:
            all_points = np.vstack(contours)
            x, y, w, h = cv2.boundingRect(all_points)
            cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 4)
            cv2.putText(img, "Robot", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        return img
    except Exception as e:
        print(f"⚠️ batch_contour_detection hatası: {e}")
        raise

# 🔧 Kontrol butonları ve tab'lar
webcam_start_button = widgets.Button(description="Akışı Başlat", button_style='info')
webcam_stop_button = widgets.Button(description="Akışı Durdur", button_style='warning')
api_start_button = widgets.Button(description="Akışı Başlat", button_style='info')
api_stop_button = widgets.Button(description="Akışı Durdur", button_style='warning')

webcam_controls = widgets.HBox([webcam_start_button, webcam_stop_button])
api_controls = widgets.HBox([api_start_button, api_stop_button])

source_tab = widgets.Tab()
source_tab.children = [
    widgets.VBox([image_dropdown]),  # Dataset
    widgets.VBox([webcam_controls]),  # Webcam
    widgets.VBox([api_controls])      # API
]
source_tab.set_title(0, "Dataset")
source_tab.set_title(1, "Webcam")
source_tab.set_title(2, "API")

# 🔄 Kaynak değişimi
def on_source_tab_change(change):
    global preview_thread
    preview_stop.set()
    if preview_thread is not None:
        preview_thread.join(timeout=1.0)
        preview_thread = None
    image_output_original.value = b""
    image_output_processed.value = b""
    source_index = change['new']
    source = {0: "Dataset", 1: "Webcam", 2: "API"}[source_index]
    print(f"Kaynak değişti: {source}")
    if source == "Dataset":
        image_dropdown.disabled = False
        selected_img = image_dropdown.value
        if selected_img and selected_img != "Görüntü yok":
            path = os.path.join(dataset_folder, selected_img)
            img = cv2.imread(path)
            if img is not None:
                show_image(img, target="original")
    else:
        image_dropdown.disabled = True

source_tab.observe(on_source_tab_change, names='selected_index')
image_dropdown.observe(lambda c: on_source_tab_change({'new': 0}), names='value')

# 🟢 Akışı başlat (Webcam)
def on_webcam_start_clicked(b):
    global preview_thread
    preview_stop.set()
    if preview_thread is not None:
        preview_thread.join(timeout=1.0)
        preview_thread = None
    preview_stop.clear()
    preview_thread = threading.Thread(target=start_webcam_preview, daemon=True)
    preview_thread.start()

webcam_start_button.on_click(on_webcam_start_clicked)

# 🟠 Akışı durdur (Webcam)
def on_webcam_stop_clicked(b):
    global preview_thread
    preview_stop.set()
    if preview_thread is not None:
        preview_thread.join(timeout=1.0)
        preview_thread = None
    image_output_original.value = b""
    image_output_processed.value = b""

webcam_stop_button.on_click(on_webcam_stop_clicked)

# 🟢 Akışı başlat (API)
def on_api_start_clicked(b):
    global preview_thread
    preview_stop.set()
    if preview_thread is not None:
        preview_thread.join(timeout=1.0)
        preview_thread = None
    preview_stop.clear()
    preview_thread = threading.Thread(target=start_api_preview, daemon=True)
    preview_thread.start()

api_start_button.on_click(on_api_start_clicked)

# 🟠 Akışı durdur (API)
def on_api_stop_clicked(b):
    global preview_thread
    preview_stop.set()
    if preview_thread is not None:
        preview_thread.join(timeout=1.0)
        preview_thread = None
    image_output_original.value = b""
    image_output_processed.value = b""

api_stop_button.on_click(on_api_stop_clicked)

# 🟢 Görüntü işle
def on_process_clicked(b):
    global last_frame, preview_thread
    preview_stop.set()
    if preview_thread is not None:
        preview_thread.join(timeout=1.0)
        preview_thread = None
    image_output_original.value = b""
    image_output_processed.value = b""

    selected_tab = tab.selected_index
    print(f"Seçili sekme: {selected_tab}")
    if selected_tab == 0:
        threshold = params_wb.children[0].value
        min_area = params_wb.children[1].value
        clahe_clip = params_wb.children[2].value
        stripe_ratio = params_wb.children[3].value
    elif selected_tab == 1:
        threshold = params_simple.children[0].value
        min_area = params_simple.children[1].value
        clahe_clip = params_simple.children[2].value
        kernel_size = params_simple.children[3].value
    elif selected_tab == 2:
        threshold = params_batch.children[0].value
        clahe_clip = params_batch.children[1].value
        kernel_size = params_batch.children[2].value

    source_index = source_tab.selected_index
    source = {0: "Dataset", 1: "Webcam", 2: "API"}[source_index]
    print(f"Seçili kaynak: {source}")
    img = None
    if source == "Dataset":
        selected_img = image_dropdown.value
        if selected_img == "Görüntü yok":
            show_error("Lütfen bir görüntü seçin.")
            return
        path = os.path.join(dataset_folder, selected_img)
        img = cv2.imread(path)
        print(f"Dataset görüntüsü yüklendi: {path}, boyutu: {img.shape if img is not None else 'Yok'}")
    elif source in ["Webcam", "API"]:
        if last_frame is not None:
            img = last_frame.copy()
            print(f"{source} son kare alındı, boyutu: {img.shape}")
        else:
            show_error("Görüntü alınamadı. Kaynağı kontrol edin.")
            return

    if img is None:
        show_error("Görüntü alınamadı.")
        return

    # Orijinal görüntüyü göster
    show_image(img, target="original")

    # Görüntüyü işle
    img_copy = img.copy()
    img_resized, gray = preprocess_image(img_copy, clahe_clip)
    print(f"Ön işlem: resized: {img_resized.shape}, gray: {gray.shape}")

    if selected_tab == 0:
        result = white_black_stripe_detection(img_resized, gray, threshold, min_area, stripe_ratio)
    elif selected_tab == 1:
        result = simple_contour_detection(img_resized, gray, threshold, min_area, kernel_size)
    elif selected_tab == 2:
        result = batch_contour_detection(img_resized, gray, threshold, kernel_size)

    # İşlenmiş görüntüyü göster
    show_image(result, target="processed")

process_button.on_click(on_process_clicked)

# 📺 Arayüz
left_panel = widgets.VBox([ 
    widgets.HTML("<h3>Kaynak|Parametre Seçimi</h3>"),  # Başlık
    source_tab,
    widgets.HTML("<br>"),  # Parametreler ile tab arasına boşluk
    tab,
    widgets.HTML("<br>"),  # Parametreler ile buton arasına boşluk 
    process_button
], layout={"padding": "10px", "overflow": "hidden", "max_width": "400px"})
right_panel = widgets.VBox([
    widgets.HTML("<h3>Orijinal | İşlenmiş</h3>"),  # Başlık
    image_container
], layout={"padding": "5px", "margin": "0 10px", "overflow": "hidden", "max_width": f"{DISPLAY_WIDTH*2+20}px"})
ui = widgets.HBox([left_panel,widgets.HTML("<br>"),widgets.HTML("<br>"),widgets.HTML("<br>"),widgets.HTML("<br>"), right_panel], layout={"overflow": "hidden", "max_width": f"{DISPLAY_WIDTH*2+150}px"})
display(ui)

In [2]:
import cv2
import numpy as np
from collections import deque
import ipywidgets as widgets
from IPython.display import display
import requests
from PIL import Image, ImageDraw, ImageFont
from io import BytesIO
import os
import threading
import time

# Veri seti klasörü ve video dosyalarını listele
dataset_folder = "dataset_videos"
os.makedirs(dataset_folder, exist_ok=True)
video_files = [f for f in os.listdir(dataset_folder) if f.endswith((".mp4", ".avi"))]

# Global değişkenler
running = False
cap = None
stop_event = threading.Event()
output_lock = threading.Lock()
last_frame = None
prev_center = None

# Widget'lar
source_dropdown = widgets.Dropdown(
    options=["Video", "Webcam", "API"],
    description="Kaynak:",
    style={'description_width': '100px'},
    layout={'width': '300px'}
)
video_dropdown = widgets.Dropdown(
    options=video_files if video_files else ["Video bulunamadı"],
    description="Video:",
    disabled=True,
    style={'description_width': '100px'},
    layout={'width': '300px'}
)

# Parametre slider'ları
history_slider = widgets.IntSlider(
    value=100, min=50, max=500, step=10,
    description="Geçmiş Kare Sayısı:",
    style={'description_width': '100px'},
    layout={'width': '300px'}
)
varThreshold_slider = widgets.FloatSlider(
    value=16.0, min=5.0, max=50.0, step=1.0,
    description="Değişim Eşiği:",
    style={'description_width': '100px'},
    layout={'width': '300px'}
)
minArea_slider = widgets.IntSlider(
    value=500, min=100, max=2000, step=50,
    description="Min Nesne Boyutu:",
    style={'description_width': '100px'},
    layout={'width': '300px'}
)
motion_threshold_slider = widgets.FloatSlider(
    value=0.5, min=0.1, max=1.0, step=0.05,
    description="Hareket Eşiği:",
    style={'description_width': '100px'},
    layout={'width': '300px'}
)

# Slider'ları alt alta düzenle
sliders_box = widgets.VBox([
    history_slider,
    varThreshold_slider,
    minArea_slider,
    motion_threshold_slider
], layout={'padding': '5px'})

# Başlat ve durdur butonları
start_button = widgets.Button(description="Başlat", button_style='success')
stop_button = widgets.Button(description="Durdur", button_style='danger', disabled=True)

# Video gösterimi için widget
DISPLAY_WIDTH, DISPLAY_HEIGHT = 450, 338  # Net görüntü boyutları
video_output = widgets.Image(
    layout={"border": "2px solid gray", "width": f"{DISPLAY_WIDTH}px", "height": f"{DISPLAY_HEIGHT}px"}
)

# API URL
API_URL = "http://127.0.0.1:8000/api/video-feed/"

# Geçmiş veriler için deque
direction_history = deque(maxlen=10)
angle_history = deque(maxlen=10)

def get_font():
    """Türkçe karakterleri destekleyen fontu al"""
    try:
        # Windows'ta Arial fontunu kullan
        font_path = "C:/Windows/Fonts/arial.ttf"
        if os.path.exists(font_path):
            return ImageFont.truetype(font_path, 20)
        # Varsayılan olarak PIL'in yükleyebileceği bir font
        return ImageFont.load_default()
    except Exception as e:
        print(f"⚠️ Font yükleme hatası: {str(e)}. Varsayılan font kullanılıyor.")
        return ImageFont.load_default()

def show_frame(frame, direction, angle, confidence):
    """Kareyi Jupyter widget'ında göster, metni PIL ile yaz"""
    try:
        # OpenCV karesini PIL görüntüsüne çevir
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(img_rgb)
        draw = ImageDraw.Draw(pil_image)
        font = get_font()
        
        # Türkçe karakterli metni yaz
        draw.text((30, 30), f"Yön: {direction}", fill=(0, 255, 0), font=font)
        draw.text((30, 60), f"Açı: {angle:.1f}", fill=(0, 255, 0), font=font)
        draw.text((30, 90), f"Güven: {confidence:.2f}", fill=(0, 255, 0), font=font)
        
        # Görüntüyü resize et ve JPEG'e çevir
        img_resized = pil_image.resize((DISPLAY_WIDTH, DISPLAY_HEIGHT))
        buffered = BytesIO()
        img_resized.save(buffered, format="JPEG")
        video_output.value = buffered.getvalue()
    except Exception as e:
        print(f"⚠️ Görüntü gösterme hatası: {str(e)}")

def detect_moving_object(frame, bg_subtractor, min_area):
    """Hareketli nesneyi algıla ve dikdörtgen içine al"""
    fg_mask = bg_subtractor.apply(frame)
    _, thresh = cv2.threshold(fg_mask, 25, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    max_area = 0
    max_contour = None
    for contour in contours:
        area = cv2.contourArea(contour)
        if area > max_area and area > min_area:
            max_area = area
            max_contour = contour
    
    if max_contour is not None:
        x, y, w, h = cv2.boundingRect(max_contour)
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        center = (x + w // 2, y + h // 2)
        return frame, center
    return frame, None

def estimate_object_motion(prev_center, current_center, threshold):
    """Nesnenin hareket yönünü tahmin et"""
    if prev_center is None or current_center is None:
        return None, 0.0, 0.0
    
    dx = current_center[0] - prev_center[0]
    dy = current_center[1] - prev_center[1]
    angle = np.arctan2(dy, dx) * 180 / np.pi if (dx != 0 or dy != 0) else 0.0
    confidence = np.sqrt(dx**2 + dy**2) / 10.0
    
    if confidence < 0.2 or (abs(dx) < threshold and abs(dy) < threshold):
        return None, 0.0, 0.0
    
    return (dx, dy), angle, confidence

def get_object_direction(dx, dy, confidence, threshold):
    """Nesnenin yönünü belirle"""
    if confidence < 0.2 or (abs(dx) < threshold and abs(dy) < threshold):
        return "SABİT", 0.0
    
    if abs(dx) > abs(dy):
        return ("SAĞ" if dx > 0 else "SOL"), confidence
    return ("AŞAĞI" if dy > 0 else "YUKARI"), confidence

def smooth_angle(history):
    """Açıları sinüs/kosinüs ile yumuşat"""
    if not history:
        return 0.0
    sin_sum = np.sum([np.sin(np.radians(a)) for a in history]) / len(history)
    cos_sum = np.sum([np.cos(np.radians(a)) for a in history]) / len(history)
    return np.degrees(np.arctan2(sin_sum, cos_sum))

def on_source_change(change):
    """Kaynak değiştiğinde video dropdown'ı güncelle"""
    global running, cap
    running = False
    stop_event.set()
    if cap is not None:
        try:
            cap.release()
        except Exception as e:
            print(f"⚠️ cap.release hatası: {str(e)}")
        cap = None
    video_output.value = b""
    start_button.disabled = False
    stop_button.disabled = True
    
    source = change['new']
    if source == "Video":
        video_dropdown.options = video_files if video_files else ["Video bulunamadı"]
        video_dropdown.disabled = False
    else:
        video_dropdown.disabled = True

source_dropdown.observe(on_source_change, names='value')
video_dropdown.observe(lambda c: on_source_change({'new': "Video"}), names='value')

def start_api_preview(bg_subtractor, min_area, motion_threshold):
    """API'den gelen video akışını işle ve göster"""
    global last_frame, prev_center, running
    last_update = time.time()
    
    while running and not stop_event.is_set():
        try:
            response = requests.get(API_URL, timeout=20, stream=True)
            if response.status_code == 200:
                boundary = response.headers.get('content-type').split('boundary=')[-1].encode()
                buffer = b""
                for chunk in response.iter_content(chunk_size=2048):
                    if not running or stop_event.is_set():
                        break
                    buffer += chunk
                    while b'--' + boundary in buffer:
                        try:
                            frame_data, buffer = buffer.split(b'--' + boundary, 1)
                            if b'Content-Type: image/jpeg' in frame_data:
                                img_data = frame_data.split(b'\r\n\r\n', 1)[1].rsplit(b'\r\n', 1)[0]
                                img = Image.open(BytesIO(img_data))
                                frame = np.array(img)
                                if frame.shape[-1] == 4:  # RGBA to BGR
                                    frame = cv2.cvtColor(frame, cv2.COLOR_RGBA2BGR)
                                elif frame.shape[-1] == 3:  # RGB to BGR
                                    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
                                
                                # Hareketli nesneyi algıla
                                frame, current_center = detect_moving_object(frame, bg_subtractor, min_area)
                                
                                # Hareket yönünü tahmin et
                                motion, angle, confidence = estimate_object_motion(prev_center, current_center, motion_threshold)
                                
                                if motion is None:
                                    direction = "SABİT"
                                    confidence = 0.0
                                else:
                                    dx, dy = motion
                                    direction, confidence = get_object_direction(dx, dy, confidence, motion_threshold)
                                
                                # Geçmişe kaydet
                                direction_history.append(direction)
                                angle_history.append(angle)
                                
                                # Yumuşatma
                                smoothed_direction = max(set(direction_history), key=list(direction_history).count)
                                smoothed_angle = smooth_angle(angle_history)
                                
                                # Kareyi göster
                                last_frame = frame.copy()
                                show_frame(last_frame, smoothed_direction, smoothed_angle, confidence)
                                
                                # Merkezi güncelle
                                prev_center = current_center
                                
                                # FPS kontrolü
                                time.sleep(0.033)  # ~30 FPS
                        except Exception as e:
                            print(f"⚠️ Kare işleme hatası: {e}")
                            continue
            else:
                print(f"⚠️ API yanıtı: {response.status_code}")
                break
        except requests.exceptions.RequestException as e:
            print(f"⚠️ API bağlantı hatası: {e}")
            break
    print("API akışı durduruldu.")
    video_output.value = b""
    running = False
    stop_event.clear()
    start_button.disabled = False
    stop_button.disabled = True

def process_stream():
    """Video akışını işle"""
    global running, cap, last_frame, prev_center
    bg_subtractor = cv2.createBackgroundSubtractorMOG2(
        history=history_slider.value,
        varThreshold=varThreshold_slider.value,
        detectShadows=False
    )
    
    # Parametreleri al
    min_area = minArea_slider.value
    motion_threshold = motion_threshold_slider.value
    
    # Kaynak seçimi
    source = source_dropdown.value
    
    if source == "Video":
        if not video_files:
            print("⚠️ Veri setinde video bulunamadı!")
            start_button.disabled = False
            stop_button.disabled = True
            return
        selected_video = video_dropdown.value
        if selected_video == "Video bulunamadı":
            print("⚠️ Video seçilmedi!")
            start_button.disabled = False
            stop_button.disabled = True
            return
        path = os.path.join(dataset_folder, selected_video)
        cap = cv2.VideoCapture(path)
        if not cap.isOpened():
            print(f"⚠️ Video yüklenemedi: {selected_video}")
            start_button.disabled = False
            stop_button.disabled = True
            return
    
    elif source == "Webcam":
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            print("⚠️ Webcam başlatılamadı!")
            start_button.disabled = False
            stop_button.disabled = True
            return
    
    elif source == "API":
        start_api_preview(bg_subtractor, min_area, motion_threshold)
        return
    
    # İlk kareyi al
    ret, frame = cap.read()
    if not ret:
        print("⚠️ Görüntü alınamadı!")
        cap.release()
        cap = None
        start_button.disabled = False
        stop_button.disabled = True
        return
    last_frame = frame.copy()
    show_frame(last_frame, "SABİT", 0.0, 0.0)
    
    # Hareket tahmini döngüsü
    while running and not stop_event.is_set():
        ret, frame = cap.read()
        if not ret:
            print("⚠️ Video veya webcam akışı bitti!")
            break
        
        # Hareketli nesneyi algıla
        frame, current_center = detect_moving_object(frame, bg_subtractor, min_area)
        
        # Hareket yönünü tahmin et
        motion, angle, confidence = estimate_object_motion(prev_center, current_center, motion_threshold)
        
        if motion is None:
            direction = "SABİT"
            confidence = 0.0
        else:
            dx, dy = motion
            direction, confidence = get_object_direction(dx, dy, confidence, motion_threshold)
        
        # Geçmişe kaydet
        direction_history.append(direction)
        angle_history.append(angle)
        
        # Yumuşatma
        smoothed_direction = max(set(direction_history), key=list(direction_history).count)
        smoothed_angle = smooth_angle(angle_history)
        
        # Kareyi göster
        last_frame = frame.copy()
        show_frame(last_frame, smoothed_direction, smoothed_angle, confidence)
        
        # Merkezi güncelle
        prev_center = current_center
        
        time.sleep(0.033)  # ~30 FPS hedefi
    
    # Temizlik
    if cap is not None:
        try:
            cap.release()
        except Exception as e:
            print(f"⚠️ cap.release hatası: {str(e)}")
        cap = None
    video_output.value = b""
    running = False
    stop_event.clear()
    start_button.disabled = False
    stop_button.disabled = True
    print("Hareket tahmini durduruldu.")

def start_processing(b):
    """Hareket tahminini başlat"""
    global running, prev_center
    if not running:
        running = True
        prev_center = None  # Reset center for new stream
        stop_event.clear()
        start_button.disabled = True
        stop_button.disabled = False
        threading.Thread(target=process_stream, daemon=True).start()

def stop_processing(b):
    """Hareket tahminini durdur"""
    global running, cap, prev_center
    running = False
    prev_center = None
    stop_event.set()
    if cap is not None:
        try:
            cap.release()
        except Exception as e:
            print(f"⚠️ cap.release hatası: {str(e)}")
        cap = None
    video_output.value = b""
    start_button.disabled = False
    stop_button.disabled = True
    print("Hareket tahmini durduruldu.")

# Event handler'lar
start_button.on_click(start_processing)
stop_button.on_click(stop_processing)

# Arayüzü göster
left_panel = widgets.VBox([
    widgets.HTML("<h3>Kaynak|Parametre Seçimi</h3>"),  # Başlık
    source_dropdown,
    video_dropdown,
    widgets.HTML("<br>"),
    sliders_box,
    widgets.HTML("<br>"),
    widgets.HBox([start_button, stop_button])
], layout={'padding': '10px'})

right_panel = widgets.VBox([
    widgets.HTML("<h3>Video</h3>"),
    video_output
], layout={'padding': '10px', 'margin': '0 20px'})

ui = widgets.HBox(
    [left_panel,
     widgets.HTML("<br>"),
     widgets.HTML("<br>"),
     widgets.HTML("<br>"),
     widgets.HTML("<br>"),
    right_panel])
display(ui)

⚠️ Video veya webcam akışı bitti!
Hareket tahmini durduruldu.
